In [1]:
import torch

In [4]:
class MyModelLN(torch.nn.Module):
    ######Add this to the previous setup
    #block the entire layer grouping and input that
    class Block(torch.nn.Module):
        def __init__(self, in_channels, out_channels):
            super().__init__()
            self.model = torch.nn.Sequential(
                torch.nn.Linear(in_channels, out_channels),
                torch.nn.LayerNorm(out_channels),
                torch.nn.ReLU(),
                torch.nn.Linear(out_channels, out_channels),
                torch.nn.LayerNorm(out_channels),
                torch.nn.ReLU(),
            )
            #need to add a skip connection because the output is not the same as the input
            if in_channels != out_channels: #should just be true the first time through to avoid 
                self.skip = torch.nn.Linear(in_channels, out_channels)
            else:
                self.skip = torch.nn.Identity()

        def forward(self, x):
            ##############this is what is added to be a residual self.model(x) is adding the input back [x] 
            ##############but x can't be added because size issue so add it to the model so we add self.model
            return self.skip(x) + self.model(x) 
        
    def __init__(self, layer_size=[512, 512, 512]):
        super().__init__()
        layers = []
        layers.append(torch.nn.Flatten())
        c = 128 * 128 * 3
        layers.append(torch.nn.Linear(c, 512, bias=False)) #######start with a linear layer before the block
        c = 512
        #add the layers which are the block above based on how many layers you want.
        for s in layer_size:
            layers.append(self.Block(c, s)) #input the block
            c = s
        layers.append(torch.nn.Linear(c, 102, bias=False))
        self.model = torch.nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


x = torch.randn(10, 3, 128, 128)
net = MyModelLN([512] * 4)
print(net(x))

tensor([[ 0.8503,  1.9238,  1.3556,  ..., -0.9607, -1.4069, -1.5575],
        [ 0.2782,  2.2353,  1.0638,  ..., -1.4543, -1.6617,  0.3070],
        [-0.3003,  1.7740,  0.2733,  ..., -0.8853, -0.9656, -0.7349],
        ...,
        [-0.5395,  2.6890,  0.9274,  ..., -0.7664, -0.6677, -1.4706],
        [ 0.2554,  1.2711,  1.4542,  ..., -0.7423, -0.8992, -0.6533],
        [ 0.9802,  2.3308,  1.5810,  ..., -0.3575, -0.5180, -0.6449]],
       grad_fn=<MmBackward0>)


In [ ]:
128 * 128 * 3